# NB06 — Self-Supervised Pre-Training (BYOL + MFR)

Two-phase pretraining as described in manuscript Section 2C. Phase 1 (epochs 1–2): feature-space BYOL+MFR on the MILTransformer aggregator using cached features from NB05. Phase 2 (epochs 3–4): end-to-end fine-tuning where ConvNeXt-Tiny backbone reads raw tiles and is jointly trained with the aggregator at a lower learning rate (backbone LR = 0.1 × aggregator LR). EMA teacher is updated continuously across both phases.

Total: 4 epochs, ~72 hours on RTX 4090. Outputs final aggregator weights, pretrained backbone weights, and slide embeddings used by NB07 onward.

In [ ]:
import os, sys, json, math, random, gc, platform
from pathlib import Path
from time import perf_counter
from datetime import datetime
from dataclasses import dataclass
from typing import List, Dict

WORKSPACE = Path(os.environ.get('WORKSPACE', './workspace'))
WSI_ROOT  = Path(os.environ.get('WSI_ROOT',  './data/wsi'))
FEATURES05 = WORKSPACE / 'features' / 'scale0p5'
FEATURES20 = WORKSPACE / 'features' / 'scale2p0'
LOGS    = WORKSPACE / 'logs'
WEIGHTS = WORKSPACE / 'weights'
FIGS    = WORKSPACE / 'figures'
EMBED   = WORKSPACE / 'embeddings' / 'student_final'
for p in [LOGS, WEIGHTS, FIGS, EMBED]:
    p.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
import torchvision.transforms as T
from PIL import Image
import openslide
from safetensors.torch import save_file as save_safetensors, load_file as load_safetensors

CONFIG = {
    'seed': 13,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'dtype_amp': 'float16',
    'token_budget_0p5': 1200,
    'token_budget_2p0': 400,
    'mask_frac': 0.25,
    'lambda_mfr': 0.5,
    'd_model': 768, 'n_heads': 8, 'n_layers': 6, 'ff_mult': 4, 'dropout': 0.1,
    'batch_slides': 3, 'grad_accum': 2,
    'epochs_phase1': 2,
    'epochs_phase2': 2,
    'lr': 1.5e-4,
    'lr_backbone_ratio': 0.1,
    'weight_decay': 1e-4,
    'ema_tau': 0.996,
    'warmup_steps': 500,
    'save_every_steps': 1000,
    'log_every_steps': 50,
    'resume_if_available': True,
    'export_embeddings_after_train': True,
    'aug_dropout_p': 0.1,
    'aug_noise_std': 0.02,
    'p2_tiles_per_slide': 96,
    'p2_extract_batch': 256,
}

SEED = CONFIG['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if hasattr(torch.backends, 'cudnn'):
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

DEVICE = CONFIG['device']
AMP_DTYPE = (torch.float16 if (DEVICE == 'cuda' and CONFIG['dtype_amp'] == 'float16') else
             torch.bfloat16 if (DEVICE == 'cuda' and CONFIG['dtype_amp'] == 'bfloat16') else
             torch.float32)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
_to_tensor = T.ToTensor()
_resize    = T.Resize((224, 224), interpolation=T.InterpolationMode.BILINEAR)
_normalize = T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

def to_model_tensor(img: Image.Image) -> torch.Tensor:
    if img.size != (224, 224):
        img = _resize(img)
    t = _to_tensor(img); t = _normalize(t)
    return t

class ConvNeXtTinyFeats(nn.Module):
    def __init__(self, trainable: bool = True):
        super().__init__()
        w = tvm.ConvNeXt_Tiny_Weights.DEFAULT
        m = tvm.convnext_tiny(weights=w)
        self.features = m.features
        self.gap = nn.AdaptiveAvgPool2d(1)
        for p in self.parameters():
            p.requires_grad = trainable
        if not trainable:
            self.eval()
    def forward(self, x):
        x = self.features(x)
        return self.gap(x).flatten(1)

def _collect(dir_path: Path) -> Dict[str, Path]:
    return {p.stem: p for p in dir_path.glob('*.npy')}

mp05 = _collect(FEATURES05); mp20 = _collect(FEATURES20)
common_ids = sorted(set(mp05.keys()) & set(mp20.keys()))
assert len(common_ids) > 0, 'no slides have both 0.5 and 2.0 μm features; run NB05 first'

@dataclass
class SlideRec:
    slide_id: str
    npy05: Path; meta05: Path
    npy20: Path; meta20: Path

def meta_path(npy_path: Path) -> Path:
    return npy_path.with_name(npy_path.stem + '_meta.parquet')

slides: List[SlideRec] = []
for sid in common_ids:
    p05 = mp05[sid]; p20 = mp20[sid]
    m05 = meta_path(p05); m20 = meta_path(p20)
    if m05.exists() and m20.exists():
        slides.append(SlideRec(sid, p05, m05, p20, m20))

print(json.dumps({
    'time': datetime.now().isoformat(timespec='seconds'),
    'torch': torch.__version__, 'device': DEVICE,
    'amp_dtype': str(AMP_DTYPE).split('.')[-1],
    'slides_2scale': len(slides),
    'phase1_epochs': CONFIG['epochs_phase1'],
    'phase2_epochs': CONFIG['epochs_phase2'],
}, indent=2))

_META_CACHE: Dict[Path, pd.DataFrame] = {}
def load_meta(p: Path) -> pd.DataFrame:
    if p in _META_CACHE: return _META_CACHE[p]
    df = pd.read_parquet(p)
    cols_lower = {c.lower(): c for c in df.columns}
    def pick(*names):
        for n in names:
            if n in df.columns: return n
            if n.lower() in cols_lower: return cols_lower[n.lower()]
        raise KeyError(f'missing one of {names} in {p.name}')
    xcol = pick('x'); ycol = pick('y'); lvlcol = pick('level', 'lvl')
    sccol = pick('scale_um_per_px')
    tsize = 256
    for n in ('tile_size', 'tile_px', 'size'):
        if n in df.columns:
            try: tsize = int(df[n].iloc[0])
            except Exception: pass
            break
    out = df[[xcol, ycol, lvlcol, sccol]].copy()
    out.columns = ['x', 'y', 'level', 'scale_um_per_px']
    out['tile_px'] = tsize
    _META_CACHE[p] = out
    return out

def compute_mm_xy(df: pd.DataFrame) -> np.ndarray:
    um_per_px = df['scale_um_per_px'].astype(float).to_numpy()
    mm_per_px = um_per_px / 1000.0
    cx = (df['x'].to_numpy() + df['tile_px'].to_numpy()/2.0) * mm_per_px
    cy = (df['y'].to_numpy() + df['tile_px'].to_numpy()/2.0) * mm_per_px
    return np.stack([cx, cy], axis=1).astype(np.float32)

class PositionalEncoder(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(3, d_model//2), nn.GELU(), nn.Linear(d_model//2, d_model))
    def forward(self, mmxy, scale_um):
        x = torch.cat([mmxy, scale_um], dim=-1)
        return self.proj(x)

class MFRDecoder(nn.Module):
    """masked feature reconstruction decoder.

    matches manuscript section 2C: a small decoder that reconstructs masked
    token features from the visible context. used during pretraining only;
    discarded at inference so the deployment parameter count is unaffected.
    """
    def __init__(self, d_model=768, d_dec=384, n_layers=2, n_heads=4, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(d_model, d_dec)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_dec))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        dec_layer = nn.TransformerEncoderLayer(
            d_model=d_dec, nhead=n_heads,
            dim_feedforward=4 * d_dec,
            dropout=dropout, batch_first=True, norm_first=True)
        self.dec = nn.TransformerEncoder(dec_layer, num_layers=n_layers)
        self.ln  = nn.LayerNorm(d_dec)
        self.out_proj = nn.Linear(d_dec, d_model)
    def forward(self, enc_out, mfr_index, pad_mask):
        # enc_out: [B, T, d_model]; mfr_index: [N, 2] of (batch, token); pad_mask: [B, T]
        h = self.in_proj(enc_out)
        mask_tok = self.mask_token.expand(h.size(0), h.size(1), -1)
        h = h.clone()
        h[mfr_index[:, 0], mfr_index[:, 1], :] = mask_tok[mfr_index[:, 0], mfr_index[:, 1], :]
        h = self.dec(h, src_key_padding_mask=pad_mask)
        h = self.ln(h)
        return self.out_proj(h)

class MILTransformer(nn.Module):
    def __init__(self, d_model=768, n_heads=8, n_layers=6, ff_mult=4, dropout=0.1):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=int(ff_mult * d_model),
            dropout=dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.ln  = nn.LayerNorm(d_model)
        self.pos = PositionalEncoder(d_model)
        self.proj_global = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))
        self.pred_global = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))
    def forward(self, feats, mmxy, scale_um, pad_mask):
        pos = self.pos(mmxy, scale_um)
        x = feats + pos
        x = self.enc(x, src_key_padding_mask=pad_mask)
        x = self.ln(x)
        x_for_pool = x.masked_fill(pad_mask.unsqueeze(-1), float('-inf'))
        g = x_for_pool.max(dim=1).values
        g_proj = self.proj_global(g)
        return g_proj, self.pred_global(g_proj), x

def cosine_loss(p, z):
    p = F.normalize(p, dim=-1); z = F.normalize(z.detach(), dim=-1)
    return (1.0 - (p * z).sum(dim=-1)).mean()

def ema_update(teacher, student, tau):
    with torch.no_grad():
        for pt, ps in zip(teacher.parameters(), student.parameters()):
            pt.data.mul_(tau).add_(ps.data, alpha=(1.0 - tau))

student = MILTransformer(d_model=CONFIG['d_model'], n_heads=CONFIG['n_heads'],
                         n_layers=CONFIG['n_layers'], ff_mult=CONFIG['ff_mult'],
                         dropout=CONFIG['dropout']).to(DEVICE)
teacher = MILTransformer(d_model=CONFIG['d_model'], n_heads=CONFIG['n_heads'],
                         n_layers=CONFIG['n_layers'], ff_mult=CONFIG['ff_mult'],
                         dropout=CONFIG['dropout']).to(DEVICE)
teacher.load_state_dict(student.state_dict())
for p in teacher.parameters(): p.requires_grad = False

backbone = ConvNeXtTinyFeats(trainable=True).to(DEVICE).to(memory_format=torch.channels_last)
backbone_teacher = ConvNeXtTinyFeats(trainable=False).to(DEVICE).to(memory_format=torch.channels_last)
backbone_teacher.load_state_dict(backbone.state_dict())
for p in backbone_teacher.parameters(): p.requires_grad = False

mfr_decoder = MFRDecoder(d_model=CONFIG['d_model']).to(DEVICE)

def make_optimizer(include_backbone: bool):
    student_params = [p for p in student.parameters() if p.requires_grad]
    decoder_params = [p for p in mfr_decoder.parameters() if p.requires_grad]
    groups = [{'params': student_params + decoder_params, 'lr': CONFIG['lr']}]
    if include_backbone:
        groups.append({
            'params': [p for p in backbone.parameters() if p.requires_grad],
            'lr': CONFIG['lr'] * CONFIG['lr_backbone_ratio'],
        })
    return torch.optim.AdamW(groups, weight_decay=CONFIG['weight_decay'])

def _sample(n, k):
    if n <= k: return np.arange(n, dtype=np.int64)
    return np.random.choice(n, size=k, replace=False).astype(np.int64)

def load_tokens_for_slide(rec, budget05, budget20):
    f05 = np.load(rec.npy05, mmap_mode='r')
    m05 = load_meta(rec.meta05); idx05 = _sample(f05.shape[0], budget05)
    mm05 = compute_mm_xy(m05.iloc[idx05])
    sc05 = m05['scale_um_per_px'].iloc[idx05].to_numpy(np.float32).reshape(-1, 1)
    f20 = np.load(rec.npy20, mmap_mode='r')
    m20 = load_meta(rec.meta20); idx20 = _sample(f20.shape[0], budget20)
    mm20 = compute_mm_xy(m20.iloc[idx20])
    sc20 = m20['scale_um_per_px'].iloc[idx20].to_numpy(np.float32).reshape(-1, 1)
    feats = np.concatenate([f05[idx05], f20[idx20]], axis=0).astype(np.float32)
    mmxy  = np.concatenate([mm05, mm20], axis=0).astype(np.float32)
    scl   = np.concatenate([sc05, sc20], axis=0).astype(np.float32)
    return feats, mmxy, scl

def create_augmented_view(feats_np, drop_p=0.1, noise_std=0.02):
    mask = (np.random.rand(*feats_np.shape) > drop_p).astype(np.float32)
    aug = feats_np * mask
    aug = aug + np.random.normal(0, noise_std, size=feats_np.shape).astype(np.float32)
    return aug

def make_phase1_batch(batch_recs, budget05, budget20, mask_frac):
    feats_view1 = []; feats_view2 = []
    mmxy_list = []; sc_list = []; mask_tiles = []
    for rec in batch_recs:
        f, mm, sc = load_tokens_for_slide(rec, budget05, budget20)
        Tn = f.shape[0]
        feats_view1.append(torch.from_numpy(create_augmented_view(f, CONFIG['aug_dropout_p'], CONFIG['aug_noise_std'])))
        feats_view2.append(torch.from_numpy(create_augmented_view(f, CONFIG['aug_dropout_p'], CONFIG['aug_noise_std'])))
        mmxy_list.append(torch.from_numpy(mm))
        sc_list.append(torch.from_numpy(sc))
        mcount = max(1, int(round(mask_frac * Tn)))
        mask_idx = np.random.choice(Tn, size=mcount, replace=False).astype(np.int64)
        mask_tiles.append(torch.from_numpy(mask_idx))
    Tmax = max(t.shape[0] for t in feats_view1)
    B = len(batch_recs); D = feats_view1[0].shape[1]
    f_v1 = torch.zeros(B, Tmax, D); f_v2 = torch.zeros(B, Tmax, D)
    mmxy = torch.zeros(B, Tmax, 2); scl = torch.zeros(B, Tmax, 1)
    pad = torch.ones(B, Tmax, dtype=torch.bool)
    for i in range(B):
        n = feats_view1[i].shape[0]
        f_v1[i, :n] = feats_view1[i]; f_v2[i, :n] = feats_view2[i]
        mmxy[i, :n] = mmxy_list[i]; scl[i, :n] = sc_list[i]
        pad[i, :n] = False
    mfr_index = []
    for b, idx in enumerate(mask_tiles):
        mfr_index.append(torch.stack([torch.full_like(idx, b), idx], dim=1))
    mfr_index = torch.cat(mfr_index, dim=0)
    for b_idx, m_idx in enumerate(mask_tiles):
        f_v1[b_idx, m_idx, :] = 0.0
        f_v2[b_idx, m_idx, :] = 0.0
    return {
        'feats_v1': f_v1.to(DEVICE, non_blocking=True),
        'feats_v2': f_v2.to(DEVICE, non_blocking=True),
        'mmxy': mmxy.to(DEVICE, non_blocking=True),
        'scl': scl.to(DEVICE, non_blocking=True),
        'pad': pad.to(DEVICE, non_blocking=True),
        'mfr_index': mfr_index.to(DEVICE, non_blocking=True),
    }

class CosineWarmup:
    def __init__(self, optimizer, warmup, max_steps, base_lrs):
        self.opt = optimizer
        self.warmup = warmup; self.max = max_steps
        self.base_lrs = list(base_lrs)
        self.t = 0
    def step(self):
        self.t += 1
        if self.t <= self.warmup:
            scale = self.t / max(1, self.warmup)
        else:
            p = (self.t - self.warmup) / max(1, self.max - self.warmup)
            scale = 0.5 * (1 + math.cos(math.pi * p))
        for g, base in zip(self.opt.param_groups, self.base_lrs):
            g['lr'] = base * scale
        return self.opt.param_groups[0]['lr']

LOG_CSV = LOGS / 'nb06_train_log.csv'
if not LOG_CSV.exists():
    LOG_CSV.write_text('ts,phase,epoch,step,lr,loss,loss_byol,loss_mfr,tokens_per_s,vram_gb\n', encoding='utf-8')
LOG_JL = LOGS / 'nb06_train_log.jsonl'

def log_row(d):
    d2 = d.copy(); d2['ts'] = datetime.now().isoformat(timespec='seconds')
    with open(LOG_JL, 'a', encoding='utf-8') as f:
        f.write(json.dumps(d2, ensure_ascii=False) + '\n')

def save_ckpt(tag, save_backbone=False):
    fn = WEIGHTS / f'nb06_student_{tag}.safetensors'
    save_safetensors({k: v.detach().cpu() for k, v in student.state_dict().items()}, str(fn))
    dn = WEIGHTS / f'nb06_decoder_{tag}.safetensors'
    save_safetensors({k: v.detach().cpu() for k, v in mfr_decoder.state_dict().items()}, str(dn))
    if save_backbone:
        bn = WEIGHTS / f'nb06_backbone_{tag}.safetensors'
        save_safetensors({k: v.detach().cpu() for k, v in backbone.state_dict().items()}, str(bn))
    (WEIGHTS / 'latest.txt').write_text(fn.name, encoding='utf-8')
    print(f'[SAVE] {fn.name}' + (' + backbone' if save_backbone else '') + ' + decoder')

def try_resume():
    if not CONFIG['resume_if_available']: return False
    txt = WEIGHTS / 'latest.txt'
    if not txt.exists(): return False
    ck = WEIGHTS / txt.read_text(encoding='utf-8').strip()
    if not ck.exists(): return False
    sd = load_safetensors(str(ck))
    student.load_state_dict(sd, strict=True)
    teacher.load_state_dict(sd, strict=False)
    dn = ck.parent / ck.name.replace('student_', 'decoder_')
    if dn.exists():
        mfr_decoder.load_state_dict(load_safetensors(str(dn)), strict=True)
    bb = ck.parent / ck.name.replace('student_', 'backbone_')
    if bb.exists():
        backbone.load_state_dict(load_safetensors(str(bb)), strict=True)
        backbone_teacher.load_state_dict(backbone.state_dict())
    return True

global_step = 0
resumed = try_resume()

p1_steps = CONFIG['epochs_phase1'] * (len(slides) // CONFIG['batch_slides'] + 1)
opt = make_optimizer(include_backbone=False)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
sched = CosineWarmup(opt, warmup=CONFIG['warmup_steps'], max_steps=p1_steps, base_lrs=[CONFIG['lr']])

print(f'\nPhase 1: feature-space SSL on aggregator, {CONFIG["epochs_phase1"]} epochs')
for epoch in range(1, CONFIG['epochs_phase1'] + 1):
    random.shuffle(slides)
    i = 0
    while i < len(slides):
        batch_recs = slides[i: i + CONFIG['batch_slides']]; i += CONFIG['batch_slides']
        try:
            b = make_phase1_batch(batch_recs, CONFIG['token_budget_0p5'], CONFIG['token_budget_2p0'], CONFIG['mask_frac'])
        except Exception as e:
            print(f'[SKIP] phase1 batch error: {e}'); continue
        mmxy, scl, pad, mfr_index = b['mmxy'], b['scl'], b['pad'], b['mfr_index']
        tokens_total = int((~pad).sum().item())
        opt.zero_grad(set_to_none=True)
        t0 = perf_counter()
        with torch.no_grad():
            g_t1, _, enc_t1 = teacher(b['feats_v1'], mmxy, scl, pad)
            g_t2, _, enc_t2 = teacher(b['feats_v2'], mmxy, scl, pad)
        with torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda' and AMP_DTYPE != torch.float32)):
            g_s1, g_sp1, enc_s1 = student(b['feats_v1'], mmxy, scl, pad)
            g_s2, g_sp2, enc_s2 = student(b['feats_v2'], mmxy, scl, pad)
            loss_byol = 0.5 * cosine_loss(g_sp1, g_t2) + 0.5 * cosine_loss(g_sp2, g_t1)
            bi = mfr_index
            dec_out_v1 = mfr_decoder(enc_s1, bi, pad)
            dec_out_v2 = mfr_decoder(enc_s2, bi, pad)
            d_s1m = dec_out_v1[bi[:, 0], bi[:, 1], :]
            d_s2m = dec_out_v2[bi[:, 0], bi[:, 1], :]
            t_t1m = enc_t1[bi[:, 0], bi[:, 1], :]
            t_t2m = enc_t2[bi[:, 0], bi[:, 1], :]
            loss_mfr = 0.5 * cosine_loss(d_s1m, t_t2m) + 0.5 * cosine_loss(d_s2m, t_t1m)
            loss = loss_byol + CONFIG['lambda_mfr'] * loss_mfr
        scaler.scale(loss / CONFIG['grad_accum']).backward()
        scaler.step(opt); scaler.update()
        ema_update(teacher, student, tau=CONFIG['ema_tau'])
        lr = sched.step()
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
            vram = torch.cuda.max_memory_allocated()/(1024**3)
            torch.cuda.reset_peak_memory_stats()
        else: vram = 0.0
        dt = perf_counter() - t0
        tps = tokens_total / max(dt, 1e-6)
        global_step += 1
        if global_step % CONFIG['log_every_steps'] == 0:
            print(f'[P1 E{epoch} S{global_step}] loss={loss.item():.4f} byol={loss_byol.item():.4f} mfr={loss_mfr.item():.4f} | {tps:.1f} tok/s | lr={lr:.2e} | VRAM={vram:.2f}GB')
            log_row({'phase': 1, 'epoch': epoch, 'step': global_step, 'lr': lr,
                     'loss': float(loss.item()), 'loss_byol': float(loss_byol.item()),
                     'loss_mfr': float(loss_mfr.item()), 'tps': float(tps), 'vram_gb': float(vram)})
        if global_step % CONFIG['save_every_steps'] == 0:
            save_ckpt(f'p1_e{epoch}_s{global_step}')
    save_ckpt(f'p1_e{epoch}')

print(f'\nPhase 2: end-to-end backbone+aggregator FT, {CONFIG["epochs_phase2"]} epochs')

SLIDE_INDEX_PATH = LOGS / 'slide_path_index.json'
if SLIDE_INDEX_PATH.exists():
    slide_map = json.loads(SLIDE_INDEX_PATH.read_text(encoding='utf-8'))
else:
    slide_map = {}
    for ext in ('*.svs', '*.ndpi', '*.tif', '*.mrxs', '*.scn'):
        for p in WSI_ROOT.rglob(ext):
            slide_map[p.stem] = str(p)
    SLIDE_INDEX_PATH.write_text(json.dumps(slide_map, indent=2), encoding='utf-8')

def read_raw_tiles_for_slide(rec, n_tiles):
    src = slide_map.get(rec.slide_id, None)
    if src is None or not Path(src).exists():
        return None, None, None
    m05 = load_meta(rec.meta05); m20 = load_meta(rec.meta20)
    n5 = min(int(0.75 * n_tiles), len(m05))
    n2 = min(n_tiles - n5, len(m20))
    s05 = m05.sample(n=n5, random_state=SEED) if n5 > 0 else m05.iloc[0:0]
    s20 = m20.sample(n=n2, random_state=SEED) if n2 > 0 else m20.iloc[0:0]
    osr = openslide.OpenSlide(src)
    tensors = []; mm_xy = []; sc_um = []
    for df_part, scale in [(s05, 0.5), (s20, 2.0)]:
        for _, r in df_part.iterrows():
            lvl = int(r['level']); x = int(r['x']); y = int(r['y']); tpx = int(r['tile_px'])
            ds = osr.level_downsamples[lvl]
            bx = int(round(x * ds)); by = int(round(y * ds))
            try:
                img = osr.read_region((bx, by), lvl, (tpx, tpx)).convert('RGB')
            except Exception:
                continue
            tensors.append(to_model_tensor(img))
            cx = (x + tpx/2) * (scale / 1000.0)
            cy = (y + tpx/2) * (scale / 1000.0)
            mm_xy.append([cx, cy]); sc_um.append([scale])
    osr.close()
    if not tensors:
        return None, None, None
    return (torch.stack(tensors, 0).to(memory_format=torch.channels_last),
            torch.tensor(mm_xy, dtype=torch.float32),
            torch.tensor(sc_um, dtype=torch.float32))

def encode_with_backbone(model, raw_tiles, batch_size):
    outs = []
    for i in range(0, raw_tiles.size(0), batch_size):
        chunk = raw_tiles[i:i+batch_size].to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda' and AMP_DTYPE != torch.float32)):
            outs.append(model(chunk))
    return torch.cat(outs, dim=0)

p2_steps = CONFIG['epochs_phase2'] * (len(slides) // CONFIG['batch_slides'] + 1)
opt = make_optimizer(include_backbone=True)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
sched = CosineWarmup(opt, warmup=200, max_steps=p2_steps,
                     base_lrs=[CONFIG['lr'], CONFIG['lr'] * CONFIG['lr_backbone_ratio']])

for epoch in range(1, CONFIG['epochs_phase2'] + 1):
    random.shuffle(slides)
    i = 0
    while i < len(slides):
        rec = slides[i]; i += 1
        raw, mm, sc = read_raw_tiles_for_slide(rec, CONFIG['p2_tiles_per_slide'])
        if raw is None: continue
        T_local = raw.size(0)
        opt.zero_grad(set_to_none=True)
        t0 = perf_counter()
        with torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda' and AMP_DTYPE != torch.float32)):
            feats_student = encode_with_backbone(backbone, raw, CONFIG['p2_extract_batch']).unsqueeze(0)
        with torch.no_grad():
            feats_teacher = encode_with_backbone(backbone_teacher, raw, CONFIG['p2_extract_batch']).unsqueeze(0)
        mmxy = mm.unsqueeze(0).to(DEVICE, non_blocking=True)
        scl  = sc.unsqueeze(0).to(DEVICE, non_blocking=True)
        pad  = torch.zeros(1, T_local, dtype=torch.bool, device=DEVICE)
        mcount = max(1, int(round(CONFIG['mask_frac'] * T_local)))
        mask_idx = torch.from_numpy(np.random.choice(T_local, size=mcount, replace=False).astype(np.int64)).to(DEVICE)
        with torch.amp.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda' and AMP_DTYPE != torch.float32)):
            f_in_s = feats_student.clone(); f_in_s[0, mask_idx, :] = 0.0
            g_s, g_sp, enc_s = student(f_in_s, mmxy, scl, pad)
            with torch.no_grad():
                g_t, _, enc_t = teacher(feats_teacher, mmxy, scl, pad)
            loss_byol = cosine_loss(g_sp, g_t)
            mfr_index = torch.stack([torch.zeros_like(mask_idx), mask_idx], dim=1)
            dec_out = mfr_decoder(enc_s, mfr_index, pad)
            d_s_m = dec_out[0, mask_idx, :]
            t_t_m = enc_t[0, mask_idx, :]
            loss_mfr = cosine_loss(d_s_m, t_t_m)
            loss = loss_byol + CONFIG['lambda_mfr'] * loss_mfr
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        ema_update(teacher, student, tau=CONFIG['ema_tau'])
        ema_update(backbone_teacher, backbone, tau=CONFIG['ema_tau'])
        lr = sched.step()
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
            vram = torch.cuda.max_memory_allocated()/(1024**3)
            torch.cuda.reset_peak_memory_stats()
        else: vram = 0.0
        dt = perf_counter() - t0
        tps = T_local / max(dt, 1e-6)
        global_step += 1
        if global_step % CONFIG['log_every_steps'] == 0:
            print(f'[P2 E{epoch} S{global_step}] loss={loss.item():.4f} byol={loss_byol.item():.4f} mfr={loss_mfr.item():.4f} | tiles={T_local} | {tps:.1f} tile/s | lr_agg={lr:.2e} | VRAM={vram:.2f}GB')
            log_row({'phase': 2, 'epoch': epoch, 'step': global_step, 'lr': lr,
                     'loss': float(loss.item()), 'loss_byol': float(loss_byol.item()),
                     'loss_mfr': float(loss_mfr.item()), 'tps': float(tps), 'vram_gb': float(vram)})
        if global_step % CONFIG['save_every_steps'] == 0:
            save_ckpt(f'p2_e{epoch}_s{global_step}', save_backbone=True)
        if global_step % 50 == 0:
            del raw, feats_student, feats_teacher
            gc.collect()
            if DEVICE == 'cuda': torch.cuda.empty_cache()
    save_ckpt(f'p2_e{epoch}', save_backbone=True)

print('\n[TRAIN] phase 1 + phase 2 complete')

def export_embeddings():
    txt = (WEIGHTS / 'latest.txt')
    if not txt.exists():
        print('[WARN] no latest.txt, skipping export'); return
    ckpt_name = txt.read_text(encoding='utf-8').strip()
    sd = load_safetensors(str(WEIGHTS / ckpt_name))
    student.load_state_dict(sd, strict=True); student.eval()
    bb = WEIGHTS / ckpt_name.replace('student_', 'backbone_')
    if bb.exists():
        backbone.load_state_dict(load_safetensors(str(bb)), strict=True)
    backbone.eval()
    count = 0; t0 = perf_counter()
    for rec in slides:
        outn = EMBED / f'{rec.slide_id}.npy'
        if outn.exists(): continue
        f, mm, sc = load_tokens_for_slide(rec, CONFIG['token_budget_0p5'], CONFIG['token_budget_2p0'])
        feats = torch.from_numpy(f).unsqueeze(0).to(DEVICE)
        mmxy  = torch.from_numpy(mm).unsqueeze(0).to(DEVICE)
        scl   = torch.from_numpy(sc).unsqueeze(0).to(DEVICE)
        pad   = torch.zeros(1, feats.size(1), dtype=torch.bool, device=DEVICE)
        with torch.no_grad():
            g_proj, _, _ = student(feats, mmxy, scl, pad)
        np.save(outn, g_proj.squeeze(0).cpu().numpy().astype(np.float32))
        count += 1
        if count % 200 == 0:
            print(f'[EMB] {count}/{len(slides)} saved')
    print(f'[EMB] done: {count} slides in {(perf_counter()-t0)/60:.1f} min')

if CONFIG['export_embeddings_after_train']:
    export_embeddings()

print('NB06 complete. Next: NB06C (post-pretraining diagnostics).')